# Data Exploration - Olist E-Commerce Public Dataset

## Objective
Initial examination of raw data structure and quality before loading into PostgreSQL

## Dataset Files
- `olist_customers_dataset.csv`
- `olist_geolocation_dataset.csv`
- `olist_order_items_dataset.csv`
- `olist_order_payments_dataset.csv`
- `olist_order_reviews_dataset.csv`
- `olist_orders_dataset.csv`
- `olist_products_dataset.csv`
- `olist_sellers_dataset.csv`
- `product_category_name_translation.csv`

## Checks Performed
1. Row and column counts for each table
2. Data types of each column
3. NULL & Empty value counts
4. Duplicate row counts
5. Sample data preview

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown

# CSV paths are relative to the working directory.
DATA_PATH = Path.cwd() / "datasets" / "raw" if (Path.cwd() / "datasets").is_dir() else Path.cwd().parent / "datasets" / "raw"
csv_files = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]


def explore_csv(file_path):
    """Display basic structure and quality checks; return the loaded DataFrame."""
    df = pd.read_csv(file_path)
    
    display(Markdown(f"## {file_path.name}"))
    display(Markdown(f"**Shape:** {df.shape[0]:,} rows Ã— {df.shape[1]} columns"))

    display(Markdown("### Data Types"))
    display(df.dtypes.to_frame(name='Type'))

    display(Markdown("### NULL & Empty Values"))
    # Count missing values.
    null_counts = df.isnull().sum()

    # Count blank strings in text columns.
    empty_counts = df.apply(
        lambda col: (col.astype(str).str.strip() == '').sum() if col.dtype == 'object' else 0
    )

    total_missing = null_counts + empty_counts
    missing_pct = (total_missing / len(df) * 100).round(2)

    missing_summary = pd.DataFrame({
        'NULL_Count': null_counts,
        'Empty_String_Count': empty_counts,
        'Total_Missing': total_missing,
        'Missing_Percentage': missing_pct
    })
    # Show columns with missing values.
    missing_summary = missing_summary[missing_summary['Total_Missing'] > 0]
    
    if len(missing_summary) > 0:
        display(missing_summary)
    else:
        display(Markdown("*No NULL or empty values found*"))

    # Count duplicate rows.
    duplicates = df.duplicated().sum()
    display(Markdown(f"**Duplicate Rows:** {duplicates:,} ({duplicates/len(df)*100:.2f}%)"))
    
    display(Markdown("### First 3 Rows"))
    display(df.head(3))
    
    display(Markdown("---"))
    
    return df


# Store loaded DataFrames for later use.
dataframes = {}

for csv_file in csv_files:
    file_path = DATA_PATH / csv_file
    if file_path.exists():
        table_name = csv_file.replace('.csv', '')
        dataframes[table_name] = explore_csv(file_path)
    else:
        display(Markdown(f"**File not found:** {csv_file}"))

display(Markdown("<div style='text-align: center'><h2>EXPLORATION COMPLETE</h2></div>"))